# Modelo convolucional 1D v2 (U-Net residual)

Versión mejorada del modelo convolucional 1D. Mantiene el planteamiento de superresolución del modelo base (entrada interpolada sobre la malla de SDSS y conexión residual global) e incorpora tres mejoras:

1. **Pérdida ponderada por la varianza inversa de SDSS** (`y_ivar`): los píxeles ruidosos del espectro objetivo pesan menos y los píxeles inválidos (`ivar = 0`) se enmascaran, de modo que la red aprende el espectro subyacente y no la realización concreta del ruido.
2. **Canal de entrada adicional con el error de flujo de Gaia** (`X_err`): la red sabe qué zonas de su entrada son fiables.
3. **Mejora de datos con ruido realista**: en cada época se perturba el flujo de Gaia con ruido gaussiano de amplitud igual a su error observacional, lo que multiplica de forma efectiva el conjunto de entrenamiento.

La arquitectura pasa de una pila de bloques residuales a una **U-Net 1D** con bloques residuales y atención de canal (*squeeze-and-excitation*). Con tres niveles de submuestreo, el campo receptivo cubre todo el espectro, por lo que la red puede corregir tanto el detalle de las líneas como el desajuste de continuo a gran escala entre Gaia y SDSS.


In [ ]:
import numpy as np
from pathlib import Path

# Ruta relativa al notebook para que funcione en cualquier equipo.
# El nombre del fichero puede llevar la fecha de generación como prefijo.
data_dir = Path("../data/splits")
candidates = (
    sorted(data_dir.glob("*processed_data_no_duplicates.npz"))
    + sorted(data_dir.glob("*processed_data.npz"))
)
processed_path = candidates[0]
print("Usando:", processed_path)

data = np.load(processed_path)
print(data.files)


In [ ]:
# Datos de entrada Gaia: flujo y error de flujo
X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

X_err_train = data["X_err_train"]
X_err_val = data["X_err_val"]
X_err_test = data["X_err_test"]

# Datos objetivo SDSS: flujo y varianza inversa
y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

y_ivar_train = data["y_ivar_train"]
y_ivar_val = data["y_ivar_val"]
y_ivar_test = data["y_ivar_test"]

# IDs de Gaia y SDSS
X_id_test = data["X_id_test"]
y_id_test = data["y_id_test"]

# Longitudes de onda
gaia_wavelength = data["gaia_wavelength"]
sdss_wavelength = data["sdss_wavelength"]

print("Gaia:", X_train.shape, "| SDSS:", y_train.shape)


Normalizamos cada par de espectros con la mediana del flujo de Gaia, igual que en el modelo base. El error de Gaia se normaliza con la misma escala para que siga siendo coherente con su flujo.

Los pesos por píxel se derivan de la varianza inversa de SDSS: al dividir el flujo por la escala `s`, la varianza queda dividida por `s²`, luego la varianza inversa del flujo normalizado es `ivar · s²`. Dentro de cada espectro normalizamos los pesos para que su media sobre los píxeles válidos sea 1: así todos los objetos contribuyen por igual a la pérdida y los pesos solo redistribuyen la importancia entre píxeles. Un recorte superior evita que unos pocos píxeles de varianza muy baja dominen el entrenamiento; los píxeles con `ivar = 0` quedan enmascarados con peso 0.


In [ ]:
# Escala por espectro: mediana del flujo de Gaia
X_scale_train = np.nanmedian(np.abs(X_train), axis=1, keepdims=True)
X_scale_val = np.nanmedian(np.abs(X_val), axis=1, keepdims=True)
X_scale_test = np.nanmedian(np.abs(X_test), axis=1, keepdims=True)

X_train_norm = X_train / X_scale_train
X_val_norm = X_val / X_scale_val
X_test_norm = X_test / X_scale_test

X_err_train_norm = X_err_train / X_scale_train
X_err_val_norm = X_err_val / X_scale_val
X_err_test_norm = X_err_test / X_scale_test

# El objetivo se normaliza con la misma escala que su entrada
y_train_norm = y_train / X_scale_train
y_val_norm = y_val / X_scale_val
y_test_norm = y_test / X_scale_test


def build_pixel_weights(y_ivar, scale, max_weight=10.0):
    # Varianza inversa del flujo normalizado
    weights = y_ivar * scale**2
    # Media 1 sobre los píxeles válidos de cada espectro
    valid = np.where(weights > 0, weights, np.nan)
    weights = weights / np.nanmean(valid, axis=1, keepdims=True)
    return np.clip(weights, 0.0, max_weight).astype(np.float32)


w_train = build_pixel_weights(y_ivar_train, X_scale_train)
w_val = build_pixel_weights(y_ivar_val, X_scale_val)
w_test = build_pixel_weights(y_ivar_test, X_scale_test)

print("Fracción de píxeles enmascarados (ivar = 0):", (w_train == 0).mean())


La interpolación lineal de Gaia sobre la malla de SDSS es una operación lineal fija, por lo que puede expresarse como una matriz de `(n_sdss, n_gaia)`. Esto permite aplicarla dentro del flujo de datos de TensorFlow, de modo que el ruido de la mejora de datos se añade sobre el espectro nativo de Gaia (donde es estadísticamente correcto) y la interpolación se hace después, en cada época.


In [ ]:
def build_interp_matrix(source_wavelength, target_wavelength):
    idx = np.searchsorted(source_wavelength, target_wavelength)
    idx = np.clip(idx, 1, len(source_wavelength) - 1)
    left = idx - 1
    right = idx
    t = (target_wavelength - source_wavelength[left]) / (
        source_wavelength[right] - source_wavelength[left]
    )
    # Fuera del rango de Gaia se mantiene el valor extremo, igual que np.interp
    t = np.clip(t, 0.0, 1.0)
    matrix = np.zeros(
        (len(target_wavelength), len(source_wavelength)), dtype=np.float32
    )
    rows = np.arange(len(target_wavelength))
    matrix[rows, left] = 1.0 - t
    matrix[rows, right] = t
    return matrix


interp_matrix = build_interp_matrix(gaia_wavelength, sdss_wavelength)

# Comprobación: la matriz reproduce np.interp
reference = np.interp(sdss_wavelength, gaia_wavelength, X_train_norm[0])
assert np.allclose(X_train_norm[0] @ interp_matrix.T, reference, atol=1e-5)


def interpolate_pair(flux_norm, err_norm):
    flux_interp = (flux_norm @ interp_matrix.T).astype(np.float32)
    err_interp = (err_norm @ interp_matrix.T).astype(np.float32)
    return flux_interp, err_interp


X_val_flux_interp, X_val_err_interp = interpolate_pair(X_val_norm, X_err_val_norm)
X_test_flux_interp, X_test_err_interp = interpolate_pair(X_test_norm, X_err_test_norm)

print(X_val_flux_interp.shape, "->", y_val_norm.shape)


Flujos de datos de entrenamiento y evaluación. El de entrenamiento añade en cada época ruido gaussiano al flujo de Gaia con la amplitud de su error observacional y después interpola; los de validación y test solo interpolan. Cada elemento produce las dos entradas del modelo (flujo interpolado y error interpolado), el objetivo y los pesos por píxel.


In [ ]:
import tensorflow as tf

batch_size = 64
interp_matrix_tf = tf.constant(interp_matrix)


def project_to_sdss(flux, err, target, weight):
    flux_interp = tf.linalg.matvec(interp_matrix_tf, flux)
    err_interp = tf.linalg.matvec(interp_matrix_tf, err)
    inputs = (flux_interp[:, None], err_interp[:, None])
    return inputs, target[:, None], weight


def augment_with_noise(flux, err, target, weight):
    noise = tf.random.normal(tf.shape(flux)) * err
    return flux + noise, err, target, weight


def as_tensor_slices(flux, err, target, weight):
    return tf.data.Dataset.from_tensor_slices((
        flux.astype(np.float32),
        err.astype(np.float32),
        target.astype(np.float32),
        weight,
    ))


train_ds = (
    as_tensor_slices(X_train_norm, X_err_train_norm, y_train_norm, w_train)
    .shuffle(4096)
    .map(augment_with_noise, num_parallel_calls=tf.data.AUTOTUNE)
    .map(project_to_sdss, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    as_tensor_slices(X_val_norm, X_err_val_norm, y_val_norm, w_val)
    .map(project_to_sdss, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    as_tensor_slices(X_test_norm, X_err_test_norm, y_test_norm, w_test)
    .map(project_to_sdss, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


Arquitectura U-Net 1D:

- **Codificador**: convolución inicial (64 filtros, núcleo 9) seguida de tres niveles con un bloque residual y submuestreo por convolución con paso 2 (64 → 96 → 128 → 160 filtros).
- **Cuello de botella**: dos bloques residuales de 160 filtros que ven el espectro completo a resolución 1/8.
- **Decodificador**: tres niveles de sobremuestreo con concatenación de la conexión del codificador correspondiente y un bloque residual.
- Cada bloque residual incluye atención de canal (*squeeze-and-excitation*).
- La convolución final de un filtro genera la corrección y la **conexión residual global** le suma el flujo interpolado, igual que en el modelo base.

Como 2666 no es múltiplo de 8, se añaden 3 puntos de relleno por cada lado (2672 = 8 · 334) y la corrección se recorta a la longitud original antes de la suma global.

La pérdida es Huber ponderada: los pesos por píxel entran como `sample_weight` temporal, de forma que la métrica `weighted_mae` refleja el error que optimiza la red y `mae` el error sin ponderar.


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

n_sdss = len(sdss_wavelength)
padding = (3, 3)


def se_block(x, filters, ratio=8):
    excite = layers.GlobalAveragePooling1D()(x)
    excite = layers.Dense(filters // ratio, activation="elu")(excite)
    excite = layers.Dense(filters, activation="sigmoid")(excite)
    excite = layers.Reshape((1, filters))(excite)
    return layers.Multiply()([x, excite])


def residual_se_block(x, filters, kernel_size=7):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="elu")(x)
    x = layers.Conv1D(filters, kernel_size, padding="same")(x)
    x = se_block(x, filters)
    x = layers.Add()([x, shortcut])
    return layers.Activation("elu")(x)


def downsample(x, filters):
    return layers.Conv1D(filters, 5, strides=2, padding="same", activation="elu")(x)


def upsample_and_merge(x, skip, filters):
    x = layers.UpSampling1D(2)(x)
    x = layers.Conv1D(filters, 5, padding="same", activation="elu")(x)
    x = layers.Concatenate()([x, skip])
    return layers.Conv1D(filters, 5, padding="same", activation="elu")(x)


flux_input = layers.Input(shape=(n_sdss, 1), name="interp_flux")
err_input = layers.Input(shape=(n_sdss, 1), name="interp_err")

x = layers.Concatenate()([flux_input, err_input])
x = layers.ZeroPadding1D(padding)(x)
x = layers.Conv1D(64, 9, padding="same", activation="elu")(x)

enc1 = residual_se_block(x, 64)      # 2672 puntos
x = downsample(enc1, 96)             # 1336
enc2 = residual_se_block(x, 96)
x = downsample(enc2, 128)            # 668
enc3 = residual_se_block(x, 128)
x = downsample(enc3, 160)            # 334

x = residual_se_block(x, 160)
x = residual_se_block(x, 160)

x = upsample_and_merge(x, enc3, 128)  # 668
x = residual_se_block(x, 128)
x = upsample_and_merge(x, enc2, 96)   # 1336
x = residual_se_block(x, 96)
x = upsample_and_merge(x, enc1, 64)   # 2672
x = residual_se_block(x, 64)

correction = layers.Conv1D(1, 5, padding="same")(x)
correction = layers.Cropping1D(padding)(correction)

# Conexión residual global: salida = flujo interpolado + corrección
outputs = layers.Add()([flux_input, correction])

model_cnn_v2 = models.Model(
    [flux_input, err_input], outputs, name="cnn_unet_1d"
)

model_cnn_v2.compile(
    optimizer=Adam(learning_rate=3e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae"],
    weighted_metrics=["mae"],
)

model_cnn_v2.summary()


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=1e-4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

history = model_cnn_v2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
epochs = range(1, len(history_dict["loss"]) + 1)

metrics = [
    ("loss", "Función de pérdida"),
    ("mae", "MAE"),
    ("weighted_mae", "MAE ponderado")
]

for metric, title in metrics:
    plt.figure(figsize=(10, 6))

    plt.plot(epochs, history_dict[metric], label=f"Train {metric}")
    plt.plot(epochs, history_dict[f"val_{metric}"], label=f"Validation {metric}")

    plt.xlabel("Época")
    plt.ylabel(metric.upper())
    plt.title(f"Evolución de {title}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
test_metrics = model_cnn_v2.evaluate(
    test_ds,
    verbose=1,
    return_dict=True
)

print(test_metrics)


In [ ]:
y_pred_norm = model_cnn_v2.predict(
    [X_test_flux_interp[..., None], X_test_err_interp[..., None]]
)[..., 0]

# Deshacemos la normalización solo con la escala de la entrada de Gaia
y_pred = y_pred_norm * X_scale_test


In [ ]:
# Mismos objetos usados en las comparativas de la memoria
for i in [123, 191]:
    plt.figure(figsize=(10, 5))
    plt.plot(sdss_wavelength, y_test[i], label="SDSS real")
    plt.plot(sdss_wavelength, y_pred[i], label="SDSS predicho")
    plt.plot(gaia_wavelength, X_test[i], label="Gaia entrada", alpha=0.8)
    plt.xlabel("Longitud de onda [Å]")
    plt.ylabel("Flujo")
    plt.title(f"Gaia source_id: {X_id_test[i]} | SDSS obj_id: {y_id_test[i]}")
    plt.legend()
    plt.show()


In [ ]:
mae = np.mean(np.abs(y_test - y_pred))
mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)

print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

# Comparación invariante a escala: cada espectro se normaliza con su propia
# mediana, de modo que el ranking refleja el error de forma y no el desajuste
# de calibración absoluta entre Gaia y SDSS
y_test_shape = y_test / np.nanmedian(np.abs(y_test), axis=1, keepdims=True)
y_pred_shape = y_pred / np.nanmedian(np.abs(y_pred), axis=1, keepdims=True)
X_test_shape = X_test / np.nanmedian(np.abs(X_test), axis=1, keepdims=True)

mae_per_object = np.mean(np.abs(y_test_shape - y_pred_shape), axis=1)
mse_per_object = np.mean((y_test_shape - y_pred_shape) ** 2, axis=1)

worst_idx = np.argsort(mse_per_object)[-10:]

for i in worst_idx:
    plt.figure(figsize=(10, 4))
    plt.plot(sdss_wavelength, y_test_shape[i], label="SDSS real")
    plt.plot(sdss_wavelength, y_pred_shape[i], label="SDSS predicho")
    plt.plot(gaia_wavelength, X_test_shape[i], label="Gaia entrada", alpha=0.8)
    plt.xlabel("Longitud de onda [Å]")
    plt.ylabel("Flujo normalizado")
    plt.title(f"Peor objeto test {i} | MSE={mse_per_object[i]:.4f} | MAE={mae_per_object[i]:.4f}")
    plt.legend()
    plt.show()


In [ ]:
best_idx = np.argsort(mse_per_object)[:10]

for i in best_idx:
    plt.figure(figsize=(10, 4))
    plt.plot(sdss_wavelength, y_test_shape[i], label="SDSS real")
    plt.plot(sdss_wavelength, y_pred_shape[i], label="SDSS predicho")
    plt.plot(gaia_wavelength, X_test_shape[i], label="Gaia entrada", alpha=0.8)
    plt.xlabel("Longitud de onda [Å]")
    plt.ylabel("Flujo normalizado")
    plt.title(f"Mejor objeto test {i} | MSE={mse_per_object[i]:.4f} | MAE={mae_per_object[i]:.4f}")
    plt.legend()
    plt.show()


In [ ]:
# Guardamos el modelo entrenado para su evaluación y puesta en servicio
model_cnn_v2.save("model-cnn-v2.keras")


Validación física con iSpec: analizamos una muestra de espectros de test comparando los parámetros estelares (Teff, log g, [M/H]) derivados del espectro SDSS real frente a los derivados del espectro predicho por la red.

Esta celda carga el modelo guardado (`model-cnn-v2.keras`) y genera sus propias predicciones, por lo que no requiere haber entrenado en esta sesión: basta con ejecutar antes las celdas de preparación de datos (carga, normalización, matriz de interpolación y flujos de datos).


In [ ]:
# Recargamos el módulo si ha habido modificaciones
%load_ext autoreload
%autoreload 2

# Funciones de iSpec (el módulo está en esta misma carpeta)
from iSpec_functions import (
    load_ispec_resources,
    analyze_sample_real_vs_pred,
    analyze_ispec_errors
)

import tensorflow as tf
from astropy.table import Table

# Cargamos el modelo entrenado y generamos las predicciones sobre test
model_cnn_v2 = tf.keras.models.load_model("model-cnn-v2.keras")

y_pred_norm = model_cnn_v2.predict(
    [X_test_flux_interp[..., None], X_test_err_interp[..., None]]
)[..., 0]

# Deshacemos la normalización solo con la escala de la entrada de Gaia
y_pred = y_pred_norm * X_scale_test

# Cargamos los recursos de iSpec
resources = load_ispec_resources()

# Cargamos la tabla de Gaia guardada en local
gaia_data_path = Path("../data/gaia_data.ecsv")

gaia_data_table = Table.read(
    gaia_data_path,
    format="ascii.ecsv"
)

# Pasamos a dataframe
gaia_data_df = gaia_data_table.to_pandas()

# Analizamos con iSpec una muestra de espectros reales frente a predichos
results_df_cnn_v2 = analyze_sample_real_vs_pred(
    y_test,
    y_pred,
    X_id_test,
    gaia_data_df,
    sdss_wavelength,
    resources,
    sample_size=250
)

error_df_cnn_v2 = analyze_ispec_errors(results_df_cnn_v2)
